# Google-Authentifizierung

Die Google-Authentifizierung ermöglicht die Anmeldung an Backstage über einen Google OAuth Client.

Das Auth-Modul authentifiziert Benutzer und ordnet die externe Google-Identität einer Backstage `User` Entity zu. Kubernetes wird nicht verwendet.


## Google Auth Provider installieren

Das Google Auth Provider Modul wird im Backstage-Backend installiert.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-auth-backend-module-google-provider


## Backend-Modul registrieren

Das Google Auth Provider Modul wird im Backstage-Backend registriert, damit die Google OAuth Endpunkte beim Start bereitgestellt werden.

Dazu patchen wir die Datei [packages/backend/src/index.ts](../../mybackstage/packages/backend/src/index.ts)


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

grep -q \
  "plugin-auth-backend-module-google-provider" packages/backend/src/index.ts \
|| sed -i "/backend.start();/i backend.add(import('@backstage/plugin-auth-backend-module-google-provider'));" packages/backend/src/index.ts

# Kontrolle
yarn why @backstage/plugin-auth-backend-module-google-provider


## Google OAuth Client erstellen

In der **Google Cloud Console**:

* Ein Projekt erstellen oder auswählen
* **Google Auth Platform → Branding** öffnen
* Den OAuth-Zustimmungsbildschirm konfigurieren
* Bei einer Anwendung im Testmodus den Google-Benutzer als Testnutzer erfassen
* **Google Auth Platform → Clients** öffnen
* **Create Client** auswählen
* Anwendungstyp **Web application** verwenden
* Als Namen beispielsweise `Backstage Google Auth` eintragen
* Unter **Authorized JavaScript origins** die unten ausgegebene Frontend-URL eintragen
* Unter **Authorized redirect URIs** die unten ausgegebene Redirect URI eintragen
* OAuth Client erstellen
* `Client ID` und `Client Secret` kopieren
* Werte in [env-platen.py](../../data/env-platen.py) als `AUTH_GOOGLE_CLIENT_ID` und `AUTH_GOOGLE_CLIENT_SECRET` eintragen

Die Redirect URI muss exakt auf den Backstage Auth Handler zeigen und darf nach `frame` keinen abschliessenden Slash enthalten.


In [ ]:
%%bash
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_PORT="3001"

echo "Google Authorized JavaScript origin:"
echo "http://${BACKSTAGE_HOSTNAME}:${BACKSTAGE_PORT}"
echo
echo "Google Authorized redirect URI:"
echo "http://localhost:7007/api/auth/google/handler/frame"


## Umgebungsvariablen prüfen

Die OAuth-Zugangsdaten werden aus `env-platen.py` geladen. Das Script zeigt nur, ob die Variablen gesetzt sind, und gibt keine Secrets aus.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
set +a

test -n "${AUTH_GOOGLE_CLIENT_ID}" \
  && echo "AUTH_GOOGLE_CLIENT_ID ist gesetzt" \
  || echo "AUTH_GOOGLE_CLIENT_ID fehlt"

test -n "${AUTH_GOOGLE_CLIENT_SECRET}" \
  && echo "AUTH_GOOGLE_CLIENT_SECRET ist gesetzt" \
  || echo "AUTH_GOOGLE_CLIENT_SECRET fehlt"


## Google Auth Provider konfigurieren

Der Google Provider wird in einer separaten Backstage-Konfigurationsdatei eingerichtet.

Der Resolver `emailMatchingUserEntityProfileEmail` vergleicht die E-Mail-Adresse des Google-Kontos mit `spec.profile.email` einer Backstage `User` Entity.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

cat > app-config.google-auth.yaml <<'EOF'
auth:
  environment: development
  providers:
    google:
      development:
        clientId: ${AUTH_GOOGLE_CLIENT_ID}
        clientSecret: ${AUTH_GOOGLE_CLIENT_SECRET}
        signIn:
          resolvers:
            - resolver: emailMatchingUserEntityProfileEmail
EOF


## Google Login im Frontend konfigurieren

Die neue Backstage Frontend-Architektur verwendet eine `SignInPageBlueprint` Extension.

Das folgende Script erstellt die Extension in `packages/app/src/extensions/googleSignInPage.tsx`.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

mkdir -p packages/app/src/extensions

cat > packages/app/src/extensions/googleSignInPage.tsx <<'EOF'
import { SignInPage } from '@backstage/core-components';
import { googleAuthApiRef } from '@backstage/core-plugin-api';
import { SignInPageBlueprint } from '@backstage/plugin-app-react';

export const googleSignInPage = SignInPageBlueprint.make({
  params: {
    loader: async () => props => (
      <SignInPage
        {...props}
        provider={{
          id: 'google-auth-provider',
          title: 'Google',
          message: 'Mit Google anmelden',
          apiRef: googleAuthApiRef,
        }}
      />
    ),
  },
});
EOF

echo "Extension erstellt:"
echo "packages/app/src/extensions/googleSignInPage.tsx"


## Frontend Extension registrieren

Das Script ergänzt den Import und fügt `googleSignInPage` am Anfang der vorhandenen `features`-Liste ein.

Vor der Änderung wird eine Sicherung von `App.tsx` erstellt. Falls die erwartete `features`-Liste nicht gefunden wird, bricht das Script ohne Änderung ab.


In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/

python3 - <<'PY'
from pathlib import Path
import shutil

path = Path("packages/app/src/App.tsx")
backup = path.with_suffix(".tsx.google-auth.bak")
text = path.read_text(encoding="utf-8")

import_line = (
    "import { googleSignInPage } "
    "from './extensions/googleSignInPage';"
)

if import_line not in text:
    lines = text.splitlines()
    last_import = max(
        (index for index, line in enumerate(lines) if line.startswith("import ")),
        default=-1,
    )
    lines.insert(last_import + 1, import_line)
    text = "\n".join(lines) + "\n"

if "googleSignInPage," not in text:
    marker = "features: ["
    if marker not in text:
        raise SystemExit(
            "Keine features-Liste gefunden. App.tsx wurde nicht verändert."
        )
    text = text.replace(
        marker,
        marker + "\n    googleSignInPage,",
        1,
    )

if not backup.exists():
    shutil.copy2(path, backup)

path.write_text(text, encoding="utf-8")
print(f"App.tsx aktualisiert. Sicherung: {backup}")
PY

grep -n "googleSignInPage" packages/app/src/App.tsx


## Backstage User Entity prüfen

Der gewählte Resolver benötigt eine `User` Entity, deren `spec.profile.email` exakt der Google-E-Mail-Adresse entspricht.

Die bestehende User-Konfiguration kann mit dem folgenden Befehl gesucht werden.


In [ ]:
%%bash
cd ~/mybackstage/

grep -R --line-number --include='*.yaml' --include='*.yml' \
  -E '^kind:[[:space:]]*User|^[[:space:]]*email:' \
  examples catalog 2>/dev/null || true


## Konfiguration prüfen

Mit `yarn backstage-cli config:print` wird die zusammengeführte und aufgelöste Backstage-Konfiguration ausgegeben, ohne die Anwendung zu starten.


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Google Auth"
export BACKSTAGE_PORT="3001"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn backstage-cli config:print \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.google-auth.yaml


## Backstage starten


In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Google Auth"
export BACKSTAGE_PORT="3001"

echo "Frontend: http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
echo "Backend:  http://localhost:7007/api/auth/google/start?env=development"

source ~/.nvm/nvm.sh
cd ~/mybackstage

yarn start \
  --config ~/mybackstage/app-config.yaml \
  --config ~/mybackstage/app-config.test.yaml \
  --config ~/mybackstage/app-config.google-auth.yaml \
  2>&1 | tee /tmp/backstage-google-auth.log


**Links**

- [Google Authentication Provider](https://backstage.io/docs/auth/google/provider/)
- [Authentication in Backstage](https://backstage.io/docs/auth/)
- [Sign-in Identities and Resolvers](https://backstage.io/docs/auth/identity-resolver/)
